## 1. Comprendre le BERT et le XLM-RoBERTa

BERT (Bidirectional Encoder Representations from Transformers) et XLM-RoBERTa sont des modèles de transformateurs pré-entraînés puissants utilisés pour une variété de tâches de traitement du langage naturel (NLP). Ils sont caractérisés par leur capacité à comprendre le contexte d'un mot dans une phrase en analysant le texte dans les deux directions (gauche à droite et droite à gauche).

- **BERT**: Développé par Google, BERT est un modèle **bidirectionnel** qui utilise un mécanisme d'attention pour analyser les relations entre tous les mots d'une phrase simultanément. Il est pré-entraîné sur de grandes quantités de texte non annoté et peut être affiné pour des tâches spécifiques comme la classification de texte, la réponse aux questions, etc.

- **XLM-RoBERTa**: Développé par Facebook AI, XLM-RoBERTa est une version **multilingue** de RoBERTa (une version optimisée de BERT). Il est pré-entraîné sur des données textuelles dans plus de 100 langues, ce qui le rend particulièrement utile pour les tâches NLP multilingues ou lorsque les données d'une langue spécifique sont limitées. Comme BERT, il est basé sur l'architecture du transformateur et peut être affiné pour diverses tâches.

Ces modèles traitent le texte en utilisant la **tokenisation**, où le texte est décomposé en unités plus petites appelées jetons. Chaque modèle a son propre tokenizer qui convertit le texte brut en identifiants numériques que le modèle peut comprendre.

In [3]:
from transformers import BertTokenizer, XLMRobertaTokenizer

print("BERT and XLM-RoBERTa tokenizers imported successfully.")

BERT and XLM-RoBERTa tokenizers imported successfully.


## 2. Tokenisation du texte

La tokenisation est le processus de conversion d'une séquence de texte en une séquence de jetons (mots, sous-mots, etc.). Les modèles de transformateurs utilisent des tokenizers pré-entraînés pour s'assurer que le texte est formaté d'une manière qu'ils peuvent comprendre.

Nous allons explorer :
- **`tokenizer()`** (API moderne équivalente à `encode_plus`) pour la tokenisation
- **`tokenizer.decode()`** pour la détokenisation
- Les champs `input_ids`, `attention_mask` et les jetons spéciaux (`[CLS]`, `[SEP]`, `<s>`, `</s>`)
- La tokenisation d'une seule phrase et de deux phrases simultanément

> **Note :** `tokenizer(text, ...)` est l'API recommandée depuis Transformers v4+. Elle est identique à `tokenizer.encode_plus(text, ...)` mais supporte le batching natif.


In [ ]:
# Initialisation des tokenizers
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
xlm_roberta_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')
# Note: variable renommée xlm_roberta_tokenizer (correction de la faute xlam → xlm)

# Phrases d'exemple
# Note: bert-base-uncased a un vocabulaire anglais, donc les phrases françaises
# seront découpées en sous-mots (subword tokenization via WordPiece)
sentence_1 = "Ceci est une phrase d'exemple pour la tokenisation."
sentence_2 = "Les modèles de transformateurs sont très puissants."

# ── Méthode 1: API callable (recommandée, équivalente à encode_plus) ──
print("=" * 60)
print("--- Tokenisation avec BertTokenizer (phrase unique) ---")
bert_single = bert_tokenizer(
    sentence_1,
    add_special_tokens=True,
    max_length=50,
    padding='max_length',
    truncation=True,
    return_attention_mask=True,
    return_tensors='pt'
)
print(f"Phrase originale : {sentence_1}")
print(f"Input IDs        : {bert_single['input_ids']}")
print(f"Attention Mask   : {bert_single['attention_mask']}")
tokens = bert_tokenizer.convert_ids_to_tokens(bert_single['input_ids'][0])
print(f"Tokens           : {tokens}")
print(f"Décodé           : {bert_tokenizer.decode(bert_single['input_ids'][0])}")

# ── Méthode 2: encode_plus() — API explicitement demandée dans les instructions ──
print("\n--- Même résultat via encode_plus() ---")
bert_encode_plus = bert_tokenizer.encode_plus(
    sentence_1,
    add_special_tokens=True,
    max_length=50,
    padding='max_length',
    truncation=True,
    return_attention_mask=True,
    return_tensors='pt'
)
print(f"Input IDs identiques: {(bert_encode_plus['input_ids'] == bert_single['input_ids']).all().item()}")

print("\n" + "=" * 60)
print("--- Tokenisation avec XLMRobertaTokenizer (phrase unique) ---")
xlm_single = xlm_roberta_tokenizer(
    sentence_1,
    add_special_tokens=True,
    max_length=50,
    padding='max_length',
    truncation=True,
    return_attention_mask=True,
    return_tensors='pt'
)
print(f"Phrase originale : {sentence_1}")
print(f"Input IDs        : {xlm_single['input_ids']}")
print(f"Attention Mask   : {xlm_single['attention_mask']}")
tokens_xlm = xlm_roberta_tokenizer.convert_ids_to_tokens(xlm_single['input_ids'][0])
print(f"Tokens           : {tokens_xlm}")
print(f"Décodé           : {xlm_roberta_tokenizer.decode(xlm_single['input_ids'][0])}")

print("\n" + "=" * 60)
print("--- Tokenisation deux phrases avec BertTokenizer ---")
# BERT sépare les deux phrases par [SEP] et utilise token_type_ids
bert_two = bert_tokenizer(
    sentence_1,
    sentence_2,
    add_special_tokens=True,
    max_length=50,
    padding='max_length',
    truncation=True,
    return_attention_mask=True,
    return_tensors='pt'
)
print(f"Phrase 1 : {sentence_1}")
print(f"Phrase 2 : {sentence_2}")
print(f"Input IDs: {bert_two['input_ids']}")
print(f"Tokens   : {bert_tokenizer.convert_ids_to_tokens(bert_two['input_ids'][0])}")
print(f"Décodé   : {bert_tokenizer.decode(bert_two['input_ids'][0])}")

print("\n" + "=" * 60)
print("--- Tokenisation deux phrases avec XLMRobertaTokenizer ---")
# XLM-RoBERTa utilise <s> et </s> comme séparateurs (pas [CLS]/[SEP])
xlm_two = xlm_roberta_tokenizer(
    sentence_1,
    sentence_2,
    add_special_tokens=True,
    max_length=50,
    padding='max_length',
    truncation=True,
    return_attention_mask=True,
    return_tensors='pt'
)
print(f"Phrase 1 : {sentence_1}")
print(f"Phrase 2 : {sentence_2}")
print(f"Input IDs: {xlm_two['input_ids']}")
print(f"Tokens   : {xlm_roberta_tokenizer.convert_ids_to_tokens(xlm_two['input_ids'][0])}")
print(f"Décodé   : {xlm_roberta_tokenizer.decode(xlm_two['input_ids'][0])}")


## 3. Préparation des données d'entrée pour le modèle

Pour préparer correctement les données d'entrée pour les modèles de transformateurs, il est crucial de comprendre les jetons spéciaux et les paramètres de tokenisation comme `max_length`, `padding`, et `truncation`. L'`attention_mask` joue un rôle essentiel en indiquant au modèle quels jetons sont réels et quels sont du remplissage.

Nous allons maintenant explorer les jetons spéciaux spécifiques à chaque tokenizer et la taille de leur vocabulaire.

In [ ]:
# Afficher les jetons spéciaux pour BertTokenizer
print("--- Jetons spéciaux pour BertTokenizer ---")
print(bert_tokenizer.special_tokens_map)
print(f"Taille du vocabulaire BERT : {bert_tokenizer.vocab_size}")

# Afficher les jetons spéciaux pour XLMRobertaTokenizer
# Correction: xlam_roberta_tokenizer → xlm_roberta_tokenizer
print("\n--- Jetons spéciaux pour XLMRobertaTokenizer ---")
print(xlm_roberta_tokenizer.special_tokens_map)
print(f"Taille du vocabulaire XLM-RoBERTa : {xlm_roberta_tokenizer.vocab_size}")

# Comparaison des jetons spéciaux entre les deux modèles
print("\n--- Comparaison des jetons spéciaux ---")
print(f"{'Rôle':<20} {'BERT':<15} {'XLM-RoBERTa':<15}")
print("-" * 50)
print(f"{'Début de séquence':<20} {'[CLS]':<15} {'<s>':<15}")
print(f"{'Fin de séquence':<20} {'[SEP]':<15} {'</s>':<15}")
print(f"{'Masque (MLM)':<20} {'[MASK]':<15} {'<mask>':<15}")
print(f"{'Inconnu':<20} {'[UNK]':<15} {'<unk>':<15}")
print(f"{'Remplissage':<20} {'[PAD]':<15} {'<pad>':<15}")

# Démonstration: tokenizer.special_tokens_map retourne un dict clé→jeton
print("\nExplication:")
print("- max_length=50 : toutes les séquences sont tronquées/paddées à 50 tokens")
print("- padding='max_length' : les séquences courtes sont complétées avec [PAD]/<pad>")
print("- truncation=True : les séquences trop longues sont coupées à max_length")
print("- attention_mask : 1 pour les vrais tokens, 0 pour les tokens [PAD] ignorés")


## 4. Chargement et exploration du jeu de données

Pour préparer nos modèles, il est essentiel de charger le jeu de données et de comprendre sa structure. Nous allons utiliser la bibliothèque `pandas` pour lire les fichiers CSV, afficher les premières lignes pour visualiser les données, et vérifier la taille du jeu de données.

In [ ]:
import pandas as pd
import os
import subprocess
import zipfile

# URL du dataset
dataset_zip_url = "https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%206/W6D1%20GenAi%20France/Basics%20of%20BERT%20and%20XLM-RoBERTa%20-%20PyTorch%20-%202.zip"

# ── Téléchargement ──
print("Téléchargement du dataset...")
os.makedirs("extracted_dataset", exist_ok=True)

# Compatible Colab (commande shell) et environnement local (subprocess)
try:
    get_ipython()  # existe seulement dans Jupyter/Colab
    import subprocess
    subprocess.run(['wget', '-q', dataset_zip_url, '-O', 'dataset.zip'], check=True)
    print("Téléchargement terminé.")
except Exception:
    import urllib.request
    urllib.request.urlretrieve(dataset_zip_url, 'dataset.zip')
    print("Téléchargement terminé (urllib).")

# ── Extraction du ZIP principal ──
with zipfile.ZipFile('dataset.zip', 'r') as z:
    z.extractall('extracted_dataset')
    all_names = z.namelist()
    print(f"Fichiers extraits : {all_names[:10]}")

# ── Détection dynamique du sous-dossier extrait ──
subdirs = [d for d in os.listdir('extracted_dataset')
           if os.path.isdir(os.path.join('extracted_dataset', d))]
if subdirs:
    full_extracted_path = os.path.join('extracted_dataset', subdirs[0])
    print(f"Dossier détecté : {full_extracted_path}")
else:
    full_extracted_path = 'extracted_dataset'

# ── Extraction des CSV internes s'ils sont zippés ──
for csv_zip in ['train.csv.zip', 'test.csv.zip']:
    zip_path = os.path.join(full_extracted_path, csv_zip)
    if os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(full_extracted_path)
        print(f"Extrait : {csv_zip}")

print(f"\nContenu du dossier '{full_extracted_path}':")
print(os.listdir(full_extracted_path))

# ── Chargement des données ──
train_path = os.path.join(full_extracted_path, 'train.csv')
test_path  = os.path.join(full_extracted_path, 'test.csv')

try:
    train_df = pd.read_csv(train_path)
    test_df  = pd.read_csv(test_path)

    print("\n" + "=" * 50)
    print("DONNÉES D'ENTRAÎNEMENT")
    print("=" * 50)
    print(f"Forme : {train_df.shape}")
    print(f"Colonnes : {list(train_df.columns)}")
    print("\n5 premières lignes:")
    print(train_df.head())
    print("\nInformations générales:")
    print(train_df.info())

    # Distribution des labels si la colonne existe
    label_col = 'label' if 'label' in train_df.columns else train_df.columns[-1]
    print(f"\nDistribution de la colonne '{label_col}':")
    print(train_df[label_col].value_counts())
    print(f"Valeurs manquantes :\n{train_df.isnull().sum()}")

    print("\n" + "=" * 50)
    print("DONNÉES DE TEST")
    print("=" * 50)
    print(f"Forme : {test_df.shape}")
    print(f"Colonnes : {list(test_df.columns)}")
    print("\n5 premières lignes:")
    print(test_df.head())

except FileNotFoundError as e:
    print(f"Erreur: fichier non trouvé → {e}")
    print(f"Contenu disponible : {os.listdir(full_extracted_path)}")
except Exception as e:
    print(f"Erreur inattendue : {e}")


## 5. Création de plis de validation croisée

La validation croisée (k-fold cross-validation) est une technique essentielle pour évaluer la performance d'un modèle de manière robuste et éviter le surapprentissage. En particulier, `StratifiedKFold` garantit que chaque pli conserve la même proportion de classes que le jeu de données original, ce qui est crucial pour les tâches de classification.

Nous allons diviser le jeu de données d'entraînement en 5 plis pour l'entraînement et la validation.

In [ ]:
from sklearn.model_selection import StratifiedKFold

# ── Détection robuste de la colonne label ──
# Le nom peut varier selon le dataset (label, category, target, class...)
possible_label_cols = ['label', 'category', 'target', 'class', 'Label', 'Category']
label_col = None
for col in possible_label_cols:
    if col in train_df.columns:
        label_col = col
        break

# Si aucun nom standard trouvé, prendre la dernière colonne non-texte
if label_col is None:
    label_col = train_df.select_dtypes(include=['int', 'float']).columns[-1]
    print(f"Colonne label détectée automatiquement : '{label_col}'")
else:
    print(f"Colonne label identifiée : '{label_col}'")

# ── Initialisation de StratifiedKFold ──
# n_splits=5 : 5 plis (80% train / 20% validation à chaque fois)
# shuffle=True : mélange aléatoire AVANT la division (évite les biais d'ordre)
# random_state=42 : reproductibilité des résultats
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Listes pour stocker indices ET DataFrames de chaque pli
train_folds_indices = []
val_folds_indices   = []
train_folds_df      = []
val_folds_df        = []

print("\n=" * 50)
print("CRÉATION DES PLIS DE VALIDATION CROISÉE (K=5)")
print("=" * 50)
print(f"Dataset total : {len(train_df)} exemples\n")

for fold, (train_idx, val_idx) in enumerate(kf.split(train_df, train_df[label_col])):
    # Stocker les indices
    train_folds_indices.append(train_idx)
    val_folds_indices.append(val_idx)

    # Stocker les DataFrames correspondants
    fold_train = train_df.iloc[train_idx].reset_index(drop=True)
    fold_val   = train_df.iloc[val_idx].reset_index(drop=True)
    train_folds_df.append(fold_train)
    val_folds_df.append(fold_val)

    # Affichage des tailles
    print(f"Pli {fold+1}:")
    print(f"  Entraînement : {len(train_idx)} exemples")
    print(f"  Validation   : {len(val_idx)} exemples")

    # Vérification de la distribution stratifiée
    train_dist = fold_train[label_col].value_counts(normalize=True).sort_index()
    val_dist   = fold_val[label_col].value_counts(normalize=True).sort_index()
    print(f"  Distribution entraînement : {dict(round(train_dist, 2))}")
    print(f"  Distribution validation   : {dict(round(val_dist, 2))}")
    print()

print("Les plis ont été créés avec succès.")
print(f"→ Indices  : train_folds_indices[0] ({len(train_folds_indices[0])} éléments)")
print(f"→ DataFrames : train_folds_df[0] shape={train_folds_df[0].shape}")
print("\nLa distribution des classes est maintenue dans chaque pli (StratifiedKFold).")


## 6. Fine-tuning des modèles BERT et XLM-RoBERTa pour la classification de texte

Maintenant que nos données sont prétraitées et divisées en plis de validation croisée, voici les étapes du fine-tuning :

1. **Dataset PyTorch personnalisé** — classe `TextClassificationDataset` héritant de `torch.utils.data.Dataset` qui gère la tokenisation à la volée et retourne `input_ids`, `attention_mask` et `labels`.

2. **Chargement du modèle** — `BertForSequenceClassification.from_pretrained(...)` ou `XLMRobertaForSequenceClassification.from_pretrained(...)` avec `num_labels=N`.

3. **Optimiseur et scheduler** — `AdamW` avec weight decay + `get_linear_schedule_with_warmup` pour faire décroître le learning rate progressivement.

4. **Boucle d'entraînement** — pour chaque pli k-fold :
   - Créer les `DataLoader` train/val
   - Entraîner N epochs, calculer la loss
   - Évaluer sur le pli de validation (accuracy, F1-score, rapport de classification)

5. **Évaluation** — `sklearn.metrics` : `accuracy_score`, `classification_report`, `f1_score`

> Cette section constitue la prochaine étape du TP. Les structures `train_folds_df` et `val_folds_df` créées à l'étape 5 sont prêtes à être utilisées.
